# Europa Waters LLC: Solar Filament CV V2 Focused Fine-tuning
## 1024px + CLAHE, inherited V1 weights, fresh optimizer

Separate from V1 and the three-fold search notebook. **Not a scored improvement.**
Attach the competition dataset and **version 1 outputs of
`gregkoja/solar-filament-cv`**, with GPU and Internet enabled.
The exact V1 checkpoint hash is verified before training.

V1 tuning loss fell from 0.18129 (epoch 18) to 0.17807 (epoch 20).
This motivates testing longer training, but does not prove that more epochs
will improve Kaggle PQ. This experiment changes resolution, contrast, batch
size, LR, and training schedule together: it is not a causal ablation.
The separate grouped-fold notebook provides controlled comparisons.

Default: **up to 40 additional epochs**, 1024px input, batch 2, fresh AdamW
at 1e-4/weight-decay .01, inner-loss patience 8/min_delta 1e-4,
ReduceLROnPlateau factor .5/patience 3. CLAHE clip limit 2/grid 8x8 is applied
after resizing, identically in all phases. Epoch zero tests the inherited
weights with this new preprocessing and can remain the best checkpoint.
It is **not exact resume of V1's optimizer**.

The **two-hour cooperative wall budget** covers fine-tuning, calibration,
and optional test inference; 10 minutes are reserved for the latter stages.
Installation, model-file discovery, dataset setup, and single in-progress
operations may add overhead. It may stop well before 40 extra epochs.
Incomplete inference never writes a partial submission.
CUDA AMP is used for model operations; Dice/BCE reductions remain float32.
No automatic parameter changes or silent OOM fallback.

V1's exact day-grouped split is reused. Training uses all annotator targets;
calibration uses the same up-to-32 tuning images as V1. V1 holdout labels
are not evaluated or used for selection. Tuning PQ is selection-biased,
not independent validation of a claimed improvement.
No video rendering, grouped-fold runs, or competition API upload occurs.

In [ ]:
%pip install -q 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy 'opencv-python-headless>=4.9,<5'

In [ ]:
import csv
import hashlib
import importlib.metadata
import itertools
import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from PIL import Image
import pycocotools.mask as mask_utils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(seed=2026, image_size=512, batch_size=8, epochs=20,
           learning_rate=3e-4, encoder='resnet18', architecture='Unet',
           pretrained=True, workers=0, tune_fraction=0.15, holdout_fraction=0.15,
           tune_image_limit=32, thresholds=[0.35, 0.5, 0.65],
           min_areas=[64, 256], contrast_thresholds=[0.025, 0.04, 0.06])
RUN_NAME = 'v2_focused_1024_clahe_from_v1'
RUN_TRAINING = False
RUN_CALIBRATION = False
SMOKE_TEST = False
SUBMIT_TO_KAGGLE = False
DATA_ROOT_OVERRIDE = None
CHECKPOINT_INPUT_DIR = None  # Path to an attached experiment folder for inference-only reproduction.
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else Path.cwd() / 'cv_outputs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else OUTPUT_DIR
CHECKPOINT_PATH = INPUT_DIR / 'best_model.pt'
COMPETITION = 'filament-segmentation-2026'
DATASET_NAME = 'MAGFiLO_1.0_Kaggle_2026'
ANNOTATION_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
FULL_SHAPE = (2048, 2048)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('WARNING: CPU selected. Enable a Kaggle GPU before full training.')
if CFG['image_size'] % 32 or CFG['batch_size'] < 2 or CFG['epochs'] < 1:
    raise ValueError('Use image_size divisible by 32, batch_size >= 2, and epochs >= 1.')
print('Device:', DEVICE, 'outputs:', OUTPUT_DIR)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))


In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')

annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
print('V2 data:', len(train_paths), 'labeled JPEGs;', len(test_paths), 'test JPEGs')


In [ ]:
@lru_cache(maxsize=128)
def small_image(filename):
    with Image.open(train_lookup[filename]) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{filename}: expected 2048 x 2048 grayscale JPEG.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0

def ann_rle(annotation):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f'Invalid polygon: {annotation["id"]}')
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation['size']) != list(FULL_SHAPE):
            raise ValueError('RLE dimensions do not match the image.')
        if isinstance(segmentation['counts'], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation['counts']
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode('ascii') if isinstance(counts, str) else counts)
    else:
        raise TypeError(f'Unsupported segmentation: {annotation["id"]}')
    if mask_utils.area(rle) <= 0:
        raise ValueError(f'Empty ground-truth filament: {annotation["id"]}')
    return rle

@lru_cache(maxsize=512)
def gt_rles(image_id):
    return [ann_rle(a) for a in annotations_by_id[image_id]]

def union_mask(rles):
    if not rles:
        return np.zeros(FULL_SHAPE, dtype=np.uint8)
    return mask_utils.decode(mask_utils.merge(rles))

@lru_cache(maxsize=128)
def small_target(image_id):
    full_mask = union_mask(gt_rles(image_id))
    resized = Image.fromarray(full_mask).resize((CFG['image_size'], CFG['image_size']), Image.Resampling.NEAREST)
    return np.asarray(resized, dtype=np.float32)

class FilamentDataset(Dataset):
    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = small_image(record['file_name'])
        target = small_target(record['id'])
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        x = torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None]
        y = torch.from_numpy(np.ascontiguousarray(target))[None]
        return x, y



In [ ]:
def build_model(config, pretrained=False):
    constructors = {'Unet': smp.Unet, 'FPN': smp.FPN}
    if config['architecture'] not in constructors:
        raise ValueError('Supported architectures: Unet, FPN.')
    return constructors[config['architecture']](encoder_name=config['encoder'],
        encoder_weights='imagenet' if pretrained else None, in_channels=1, classes=1, activation=None)

def segmentation_loss(logits, target):
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice



In [ ]:
@torch.inference_mode()
def predict_probability(path):
    with Image.open(path) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{path.name}: invalid image mode or dimensions.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np.ascontiguousarray((array - 0.5) / 0.5))[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f'Nonfinite prediction: {path.name}')
    return probability

def full_probability(probability):
    resized = Image.fromarray(np.asarray(probability, dtype=np.float32)).resize(
        (2048, 2048), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def contrast_foreground(filename, threshold):
    image = small_image(filename)
    disk = ndi.binary_erosion(image > 0.08, iterations=max(1, CFG['image_size']//128))
    background = ndi.gaussian_filter(image, sigma=CFG['image_size']/32)
    return disk & ((background - image) > threshold)

def instance_rles(probability, threshold, min_area):
    if min_area < 1 or not 0 <= threshold <= 1:
        raise ValueError('Invalid threshold or minimum area.')
    binary = full_probability(probability) >= threshold
    labels, count = ndi.label(binary, structure=np.ones((3, 3), dtype=np.uint8))
    areas = np.bincount(labels.ravel())
    result = []
    for label in range(1, count+1):
        if areas[label] >= min_area:
            mask = np.asfortranarray(labels == label, dtype=np.uint8)
            result.append(mask_utils.encode(mask))
    return result

def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0]*n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        # Maximum cardinality first, then overlap; greedy matching can lose true positives.
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched); fp = n_pred-tp; fn = n_true-tp
    denominator = tp + 0.5*fp + 0.5*fn
    pq = sum(matched)/denominator if denominator else 1.0
    pred_union, true_union = union_mask(predicted).astype(bool), union_mask(truth).astype(bool)
    intersection = np.count_nonzero(pred_union & true_union)
    union = np.count_nonzero(pred_union | true_union)
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2*intersection/pixels if pixels else 1.0,
                iou=intersection/union if union else 1.0, tp=tp, fp=fp, fn=fn,
                split_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=0) > 1)),
                merge_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=1) > 1)))

def evaluate(names, parameters, probability_cache=None):
    rows = []
    for filename in names:
        if parameters['method'] == 'unet':
            probability = (probability_cache[filename] if probability_cache is not None
                           else predict_probability(train_lookup[filename]))
            threshold = parameters['threshold']
        elif parameters['method'] == 'contrast':
            probability = contrast_foreground(filename, parameters['contrast_threshold']).astype(np.float32)
            threshold = 0.5
        else:
            raise ValueError('Unknown segmentation method.')
        predicted = instance_rles(probability, threshold, parameters['min_area'])
        for record in records_by_name[filename]:
            rows.append(dict(filename=filename, image_id=record['id'],
                             **local_scores(predicted, gt_rles(record['id']))))
    if not rows:
        raise ValueError('Evaluation set is empty.')
    # Average annotators per image, then physical images; do not overweight triply labeled images.
    grouped = defaultdict(list)
    for row in rows: grouped[row['filename']].append(row)
    metrics = {key: float(np.mean([np.mean([r[key] for r in group]) for group in grouped.values()]))
               for key in ['pq', 'dice', 'iou', 'tp', 'fp', 'fn', 'split_relations', 'merge_relations']}
    return metrics, rows


In [ ]:
import gc
import cv2

FOCUS = dict(
    image_size=1024, batch_size=2, learning_rate=1e-4,
    weight_decay=0.01, additional_epochs=40, patience=8, min_delta=1e-4,
    hours=2.0, reserve_seconds=600, clahe=True, clip_limit=2.0, tile_grid=8,
    calibration_images=32, thresholds=[0.35, 0.5, 0.65], min_areas=[64, 256],
)
BASELINE_HASH = "92ffa38c11b6d6649027b49e6a187dcc5c457b7155e5974fb2adf4ea0579cd91"
BASELINE_OVERRIDE = None
RUN_FINE_TUNING = True
RUN_TEST_EXPORT = True
USE_MIXED_PRECISION = True
ALLOW_CPU_SMOKE = False


class FocusBudgetExpired(RuntimeError):
    pass


def focus_deadline(deadline):
    if time.monotonic() >= deadline:
        raise FocusBudgetExpired("Two-hour experiment allowance reached.")


def find_baseline():
    if BASELINE_OVERRIDE is not None:
        candidates = [Path(BASELINE_OVERRIDE) / "best_model.pt"]
    else:
        candidates = sorted(Path("/kaggle/input").rglob("best_model.pt"))
    for path in candidates:
        if hashlib.sha256(path.read_bytes()).hexdigest() == BASELINE_HASH:
            return path.parent
    raise FileNotFoundError(
        "Matching V1 checkpoint not found. Attach gregkoja/solar-filament-cv "
        "version 1 outputs, or set BASELINE_OVERRIDE to its experiment folder."
    )


def focus_image(path):
    with Image.open(path) as image:
        if image.mode != "L" or image.size != (2048, 2048):
            raise ValueError(f"{path.name}: expected 2048x2048 grayscale image.")
        array = np.asarray(image.resize(
            (FOCUS["image_size"], FOCUS["image_size"]), Image.Resampling.BILINEAR
        ))
    if FOCUS["clahe"]:
        array = cv2.createCLAHE(
            clipLimit=FOCUS["clip_limit"],
            tileGridSize=(FOCUS["tile_grid"], FOCUS["tile_grid"]),
        ).apply(array)
    return np.ascontiguousarray(array, dtype=np.float32) / 255.0


class FocusDataset(Dataset):
    def __init__(self, names, augment=False):
        self.entries = [record for name in names for record in records_by_name[name]]
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = focus_image(train_lookup[record["file_name"]])
        target = np.asarray(Image.fromarray(union_mask(gt_rles(record["id"]))).resize(
            (FOCUS["image_size"], FOCUS["image_size"]), Image.Resampling.NEAREST
        ), dtype=np.float32)
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        return (
            torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None],
            torch.from_numpy(np.ascontiguousarray(target))[None],
        )


@torch.inference_mode()
def focus_probability(path):
    array = focus_image(path)
    tensor = torch.from_numpy((array - 0.5) / 0.5)[None, None].to(DEVICE)
    probability = model(tensor).float().sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f"Nonfinite prediction for {path.name}")
    return probability


def focus_save(path, payload):
    temporary = path.with_suffix(".tmp.pt")
    torch.save(payload, temporary)
    temporary.replace(path)


def run_focused():
    global model, checkpoint, CHECKPOINT_PATH, CHECKPOINT_SHA256, split_names, predict_probability
    if DEVICE.type != "cuda" and not ALLOW_CPU_SMOKE:
        raise RuntimeError("Enable a Kaggle GPU for this full-resolution training run.")
    if FOCUS["image_size"] % 32 or FOCUS["batch_size"] < 2:
        raise ValueError("Resolution must be divisible by 32 and batch size at least two.")
    if FOCUS["hours"] <= 0 or FOCUS["reserve_seconds"] >= FOCUS["hours"] * 3600:
        raise ValueError("Training budget must exceed the scoring reserve.")
    baseline_dir = find_baseline()
    baseline = torch.load(baseline_dir / "best_model.pt", map_location="cpu", weights_only=True)
    calibration = json.loads((baseline_dir / "calibration.json").read_text())
    split = json.loads((baseline_dir / "split_manifest.json").read_text())
    if (baseline["smoke_test"] or baseline["annotation_sha256"] != annotation_sha256
            or calibration["checkpoint_sha256"] != BASELINE_HASH
            or calibration["split_manifest"] != split):
        raise ValueError("V1 checkpoint/calibration/split provenance mismatch.")
    split_names = split["splits"]
    if set(n for names in split_names.values() for n in names) != set(records_by_name):
        raise ValueError("V1 split does not match the labeled dataset.")
    for a, b in itertools.combinations(split_names, 2):
        if {n[:8] for n in split_names[a]} & {n[:8] for n in split_names[b]}:
            raise ValueError("Observation-day leakage in V1 split.")
    plan = dict(
        settings=FOCUS, baseline_checkpoint_sha256=BASELINE_HASH,
        annotation_sha256=annotation_sha256, baseline_epoch=baseline["epoch"],
        smoke_test=SMOKE_TEST, split_manifest=split, mixed_precision=USE_MIXED_PRECISION,
    )
    plan_path = OUTPUT_DIR / "focused_plan.json"
    if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
        raise ValueError("Output directory has a different plan; use a new RUN_NAME.")
    write_json(plan_path, plan)
    write_json(OUTPUT_DIR / "split_manifest.json", split)
    CHECKPOINT_PATH = OUTPUT_DIR / "best_model.pt"
    started = time.monotonic()
    overall_deadline = started + FOCUS["hours"] * 3600
    training_deadline = overall_deadline - FOCUS["reserve_seconds"]
    model = build_model(baseline["config"], pretrained=False).to(DEVICE)
    model.load_state_dict(baseline["state_dict"], strict=True)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=FOCUS["learning_rate"], weight_decay=FOCUS["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )
    amp = USE_MIXED_PRECISION and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    loaders = {}
    for phase in ["train", "tune"]:
        dataset = FocusDataset(split_names[phase], augment=phase == "train")
        loaders[phase] = DataLoader(
            dataset, batch_size=FOCUS["batch_size"], shuffle=phase == "train",
            num_workers=0, drop_last=phase == "train" and len(dataset) % FOCUS["batch_size"] == 1,
        )
        if not len(loaders[phase]):
            raise ValueError("Too few entries for training/validation.")

    def phase_loss(phase, deadline):
        model.train(phase == "train")
        total, count = 0.0, 0
        with torch.set_grad_enabled(phase == "train"):
            for images, targets in tqdm(loaders[phase], desc=phase):
                focus_deadline(deadline)
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=amp):
                    logits = model(images)
                # Float32 loss reductions avoid half-precision overflow on 1024px masks.
                loss = segmentation_loss(logits.float(), targets.float())
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Nonfinite {phase} loss.")
                if phase == "train":
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                total += loss.item() * len(images); count += len(images)
        return total / count

    history, best, significant_best, stale = [], float("inf"), float("inf"), 0
    status = "inference_only"
    if RUN_FINE_TUNING:
        if CHECKPOINT_PATH.exists():
            raise FileExistsError("Saved V2 weights exist. Disable RUN_FINE_TUNING to reuse, or choose a new RUN_NAME.")
        # Epoch zero evaluates the inherited weights under the NEW preprocessing.
        best = phase_loss("tune", training_deadline)
        significant_best = best

        def save_best(epoch, value):
            focus_save(CHECKPOINT_PATH, dict(
                state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                config=dict(baseline["config"], image_size=FOCUS["image_size"],
                            batch_size=FOCUS["batch_size"], learning_rate=FOCUS["learning_rate"]),
                preprocessing=FOCUS, epoch=epoch, baseline_epoch=baseline["epoch"],
                tune_loss=value, annotation_sha256=annotation_sha256,
                baseline_checkpoint_sha256=BASELINE_HASH, smoke_test=SMOKE_TEST,
            ))
        save_best(0, best)
        history.append(dict(additional_epoch=0, train_loss=None, tune_loss=best))
        write_json(OUTPUT_DIR / "training_history.json", history)
        status = "max_epochs"
        try:
            for epoch in range(1, FOCUS["additional_epochs"] + 1):
                train_loss = phase_loss("train", training_deadline)
                tune_loss = phase_loss("tune", training_deadline)
                scheduler.step(tune_loss)
                history.append(dict(
                    additional_epoch=epoch, train_loss=train_loss, tune_loss=tune_loss,
                    learning_rate=optimizer.param_groups[0]["lr"],
                ))
                if tune_loss < best:
                    best = tune_loss
                    save_best(epoch, best)
                if tune_loss < significant_best - FOCUS["min_delta"]:
                    significant_best, stale = tune_loss, 0
                else:
                    stale += 1
                write_json(OUTPUT_DIR / "training_history.json", history)
                print("FOCUSED V2:", history[-1], flush=True)
                if stale >= FOCUS["patience"]:
                    status = "early_stopped"
                    break
        except FocusBudgetExpired as error:
            print("TRAINING BUDGET:", error, "Reloading best complete checkpoint.", flush=True)
            status = "budget_truncated"
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    if (checkpoint["preprocessing"] != FOCUS or checkpoint["smoke_test"] != SMOKE_TEST
            or checkpoint["baseline_checkpoint_sha256"] != BASELINE_HASH):
        raise ValueError("Focused checkpoint differs from this experiment.")
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    del optimizer, scheduler, scaler, loaders, baseline
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    CHECKPOINT_SHA256 = hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest()
    predict_probability = focus_probability
    write_json(OUTPUT_DIR / "training_runtime.json", dict(
        status=status, seconds=time.monotonic()-started,
        completed_additional_epochs=max(0, len(history)-1),
        best_additional_epoch=checkpoint["epoch"], device=str(DEVICE),
        note="Fine-tuning with fresh optimizer, not exact optimizer-state resume.",
    ))
    names = list(calibration["tuning_filenames"])
    if not set(names) <= set(split_names["tune"]):
        raise ValueError("Baseline calibration filenames are not tuning-only.")
    names = names[:FOCUS["calibration_images"]]
    metrics_path = OUTPUT_DIR / "calibration.json"
    tuning_results = []
    try:
        cache = {}
        for name in tqdm(names, desc="V2 tuning probabilities"):
            focus_deadline(overall_deadline)
            cache[name] = focus_probability(train_lookup[name])
        for threshold, area in itertools.product(FOCUS["thresholds"], FOCUS["min_areas"]):
            focus_deadline(overall_deadline)
            parameters = dict(method="unet", threshold=threshold, min_area=area)
            scores, _ = evaluate(names, parameters, cache)
            tuning_results.append(dict(parameters=parameters, scores=scores))
        chosen = max(tuning_results, key=lambda row: row["scores"]["pq"])
        write_json(metrics_path, dict(
            selected=dict(unet=chosen["parameters"]), candidates=tuning_results,
            tuning_filenames=names, preprocessing=FOCUS, checkpoint_sha256=CHECKPOINT_SHA256,
            baseline_checkpoint_sha256=BASELINE_HASH, smoke_test=SMOKE_TEST,
            annotation_sha256=annotation_sha256,
            note="Tuning selection metrics, not an independent holdout or Kaggle score.",
        ))
        print("V2 TUNING-ONLY PQ:", chosen["scores"]["pq"], flush=True)
    except FocusBudgetExpired as error:
        print("INCOMPLETE CALIBRATION:", error, "Weights saved; export skipped.")
        write_json(OUTPUT_DIR / "incomplete.json", dict(stage="calibration", message=str(error)))
        return None
    if RUN_TEST_EXPORT:
        if SMOKE_TEST:
            print("Smoke test: competition export disabled.")
            return chosen
        destination = OUTPUT_DIR / "submission.csv"
        temporary = destination.with_suffix(".tmp.csv")
        audit, rows = [], []
        try:
            for path in tqdm(test_paths, desc="Focused V2 test"):
                focus_deadline(overall_deadline)
                instances = instance_rles(
                    focus_probability(path), chosen["parameters"]["threshold"],
                    chosen["parameters"]["min_area"],
                )
                audit.append(dict(filename=path.name, instances=len(instances)))
                for index, rle in enumerate(instances, 1):
                    counts = rle["counts"].decode("ascii")
                    mask = mask_utils.decode(rle)
                    if mask.shape != FULL_SHAPE or not mask.any():
                        raise ValueError("Invalid/empty V2 mask.")
                    if mask_utils.encode(np.asfortranarray(mask))["counts"].decode("ascii") != counts:
                        raise ValueError("V2 COCO RLE roundtrip failed.")
                    rows.append(dict(filament_id=f"{path.stem}_{index}", segmentation_rle=counts))
        except FocusBudgetExpired as error:
            print("INCOMPLETE TEST INFERENCE:", error, "No partial submission written.")
            write_json(OUTPUT_DIR / "incomplete.json", dict(stage="test", message=str(error)))
            return chosen
        zeros = [r["filename"] for r in audit if r["instances"] == 0]
        write_json(OUTPUT_DIR / "test_prediction_audit.json", dict(images=audit, zero_detection_images=zeros))
        if not rows:
            print("NOT READY: no predictions. No CSV written.")
            return chosen
        ids = [row["filament_id"] for row in rows]
        if len(set(ids)) != len(ids) or len(audit) != len(test_paths):
            raise ValueError("Duplicate IDs or incomplete test inference.")
        with temporary.open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=["filament_id", "segmentation_rle"])
            writer.writeheader(); writer.writerows(rows)
        with temporary.open(newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != ["filament_id", "segmentation_rle"] or list(reader) != rows:
                raise ValueError("Focused V2 CSV readback failed.")
        temporary.replace(destination)
        write_json(OUTPUT_DIR / "submission_manifest.json", dict(
            smoke_test=False, rows=len(rows), images=len(audit), zero_detection_images=zeros,
            parameters=chosen["parameters"], preprocessing=FOCUS,
            checkpoint_sha256=CHECKPOINT_SHA256,
            csv_sha256=hashlib.sha256(destination.read_bytes()).hexdigest(),
        ))
        if Path("/kaggle/working").is_dir():
            shutil.copyfile(destination, "/kaggle/working/submission.csv")
        if zeros:
            print("WARNING: zero-detection rows omitted; Kaggle acceptance unverified:", zeros)
        print("CSV PREPARED:", destination, "No competition upload performed.", flush=True)
    return chosen


In [ ]:
focused_result = run_focused()

## Save evidence; submit manually only after reviewing outputs
Download the entire output folder, especially best_model.pt, focused_plan.json,
split_manifest.json, calibration.json, training_history.json, and runtime.
Disable RUN_FINE_TUNING to reuse saved V2 weights within an unchanged plan.
Use a new RUN_NAME for a fresh training experiment.

The optional CSV is copied to `/kaggle/working/submission.csv` only after complete
inference and roundtrip/readback checks. Zero-detection images are audited
without fabricated masks; acceptance is determined by Kaggle.
This notebook does **not** make a competition submission automatically.
Keep V1's submitted file and score separate.

References: U-Net (doi:10.1007/978-3-319-24574-4_28), ResNet
(doi:10.1109/CVPR.2016.90), PQ (doi:10.1109/CVPR.2019.00963),
[OpenCV CLAHE](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html).

In [ ]:
history_path = OUTPUT_DIR / "training_history.json"
if history_path.is_file():
    history = json.loads(history_path.read_text())
    figure, axis = plt.subplots(figsize=(9, 4))
    axis.plot([r["additional_epoch"] for r in history],
              [r["tune_loss"] for r in history], label="V2 tuning loss")
    trained = [r for r in history if r["train_loss"] is not None]
    axis.plot([r["additional_epoch"] for r in trained],
              [r["train_loss"] for r in trained], label="V2 training loss")
    axis.set(xlabel="Additional epoch (0 = inherited weights)", ylabel="Dice + BCE loss")
    axis.legend(); figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "loss_curves.png", dpi=150)
    plt.show()
with (OUTPUT_DIR / "requirements.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)
print("Evidence saved:", OUTPUT_DIR)
